In [1]:
!nvidia-smi
%matplotlib inline

Tue Oct 28 11:21:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:31:00.0 Off |                    0 |
| N/A   30C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
import os
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from tqdm import tqdm
from typing import List, Literal, List, Dict, Any, Optional
import numpy as np
import seaborn as sns

from datasets import load_dataset
import random
import json
import re
from functools import partial
from datasets import Dataset
from copy import deepcopy
import evaluate
import nltk
from scipy.stats import ttest_ind
import string
from collections import Counter

import openai
import os
import time
import pandas as pd
import torch

from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from dotenv import load_dotenv
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [11]:
import requests

url = {"all": "https://raw.githubusercontent.com/amazon-science/GaRAGe/refs/heads/main/data/GaRAGe_benchmark.jsonl"}

for split, url in url.items():
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"GaRAGe_{split}.jsonl", "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"{split} 集下载成功 ✅")
    else:
        print(f"{split} 集下载失败 ❌，状态码: {response.status_code}")

all 集下载成功 ✅


In [2]:
with open("GaRAGe_all.jsonl", "r", encoding="utf-8") as t:
    data = [json.loads(line) for line in t]

print(len(data))

2366


In [11]:
data[0]

{'sample_id': '3e85c5e3-ac81-484f-bee3-9cba1000169f',
 'question_date': '17 December 2024',
 'grounding': [{'age': 'unknown',
   'date': 'not defined',
   'provider': 'web',
   'cite_1': ' In 2024, BlackRock spent about $28 billion on acquisitions to strengthen its offerings in the private market, a strategic move that CEO Larry Fink sees as key to positioning the asset management giant as a conduit to private capital for global infrastructure projects at the time of tightening government budgets.'},
  {'age': 'unknown',
   'date': 'not defined',
   'provider': 'web',
   'cite_2': " Reported about 4 hours ago BlackRock's significant acquisitions in 2024, including the purchase of HPS Investment Partners for $12 billion, signal a strong commitment to expanding its presence in private markets such as private credit, real estate, and infrastructure. The firm plans to leverage these acquisitions to integrate private and public investment products, and analysts suggest that BlackRock may pu

In [3]:
df = pd.read_json("GaRAGe_all.jsonl", lines=True)
print(df.shape)
df.head()

(2366, 21)


,sample_id,question_date,grounding,question,question_valid,question_false_premise,question_seeking,question_sensitive,question_type,question_complexity,...,question_popularity,evidence_relevant,evidence_correct,answer_generate,answer_related_info,answer_validate,comments,evidence_cited,question_tag,topic_tag
0,3e85c5e3-ac81-484f-bee3-9cba1000169f,17 December 2024,"[{'age': 'unknown', 'date': 'not defined', 'pr...",What strategic move did BlackRock make in 2024...,YES,NO,YES,YES,FAST-CHANGING,Simple w. condition,...,Tail,"[YES, YES, YES, YES, YES, YES, YES, YES, YES, ...","[UNKNOWN, UNKNOWN, ANSWER-THE-QUESTION, ANSWER...","In 2024, BlackRock took several strategic step...",,YES,,"[NO, NO, YES, YES, YES, YES, YES, YES, YES, YE...",web,sec
1,340106be-dc22-4c74-9c74-620bd4246d75,17 December 2024,"[{'age': '1 months and 4 days', 'date': 'not d...",How effective were Adams Resources' hedging st...,YES,NO,YES,YES,SLOW-CHANGING,Simple w. condition,...,Tail,"[YES, YES, YES, YES, YES, YES, YES, YES, NO, N...","[ANSWER-THE-QUESTION, ANSWER-THE-QUESTION, ANS...",Adams Resources & Energy's hedging strategies ...,,YES,,"[YES, YES, YES, NO, NO, NO, NO, YES, NO, NO, N...",web,sec
2,2d38b3a4-fe6e-4850-aea9-92e4832d00ef,17 December 2024,"[{'age': 'unknown', 'date': 'not defined', 'pr...",What impact will 2024's CRISPR advancements ha...,YES,NO,YES,YES,SLOW-CHANGING,Simple w. condition,...,Tail,"[YES, YES, YES, YES, YES, YES, YES, YES, YES, ...","[UNKNOWN, ANSWER-THE-QUESTION, UNKNOWN, ANSWER...",The advancements in CRISPR technology in 2024 ...,,YES,,"[NO, YES, NO, YES, NO, NO, YES, NO, NO, NO, NO...",web,sec
3,93d93bdb-de21-4f0b-b91c-1fd6c98911e2,17 December 2024,"[{'age': '1 months and 25 days', 'date': 'not ...",How do the circulating variants KP.3.1.1 and X...,YES,NO,YES,YES,SLOW-CHANGING,Comparison,...,Head,"[YES, YES, YES, YES, YES, YES, YES, YES, YES, ...","[ANSWER-THE-QUESTION, RELATED-INFORMATION, REL...",The circulating variants KP.3.1.1 and XEC diff...,,YES,,"[YES, NO, NO, NO, NO, YES, NO, NO, NO, NO, NO,...",web,arxiv
4,eb247015-1d8f-40b5-86b6-17cf2add88bb,17 December 2024,"[{'age': 'unknown', 'date': 'not defined', 'pr...",How have James Webb Telescope observations res...,YES,NO,YES,YES,SLOW-CHANGING,Post-processing heavy,...,Tail,"[YES, YES, YES, YES, YES, YES, YES, YES, YES, ...","[UNKNOWN, UNKNOWN, ANSWER-THE-QUESTION, ANSWER...",Observations from the James Webb Space Telesco...,,YES,,"[NO, NO, YES, YES, NO, YES, NO, NO, YES, YES, ...",web,arxiv


In [8]:
df['question'].tolist()[65]

'How does the Lagrange inversion formula aid in solving equations describing dynamical systems?'

In [14]:
df_filter = df[df['answer_validate']=='YES']
print(df_filter.shape)
df_filter.columns

(1939, 21)


Index(['sample_id', 'question_date', 'grounding', 'question', 'question_valid',
       'question_false_premise', 'question_seeking', 'question_sensitive',
       'question_type', 'question_complexity', 'question_category',
       'question_popularity', 'evidence_relevant', 'evidence_correct',
       'answer_generate', 'answer_related_info', 'answer_validate', 'comments',
       'evidence_cited', 'question_tag', 'topic_tag'],
      dtype='object')

In [17]:
task_relevant_data = [{"id": item["sample_id"], "request": item["question"], "answer": [item["answer_generate"]]} for item in data]
task_relevant_data_hf = Dataset.from_list(task_relevant_data)
GaRAGe_sample = task_relevant_data_hf.shuffle(seed=42).select(range(1000))
GaRAGe_sample[0]


{'id': '76cc2729-2ec6-4100-b9c1-59c0a1b3d3e9',
 'request': 'How does Amazon Cognito support passwordless authentication methods?',
 'answer': ["Amazon Cognito supports passwordless authentication by offering multiple methods that enhance user experience and security. It enables users to log in using passkeys, email one-time passwords (OTPs), and SMS OTPs, eliminating the need for traditional passwords.[cite_4][cite_12] Passkeys leverage FIDO standards and public key cryptography to provide strong, phishing-resistant authentication. Users can utilize built-in authenticators like Touch ID or Windows Hello for a seamless experience.[cite_4][cite_10]\n\nFor developers, Amazon Cognito simplifies the implementation of passwordless authentication through a developer-focused console that streamlines onboarding with wizards and use-case-specific recommendations.[cite_5][cite_12] This flexibility extends to custom authentication flows, enabling the use of passwordless authentication methods tail

In [18]:
GaRAGe_sample.to_json('GaRAGe_sample_raw.jsonl', orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 33.44ba/s]


1259827

## Using off-the-shelf Qwen3-4B for classification

In [3]:
import os
print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


In [4]:
GaRAGe_sample = load_dataset(
    "json",
    data_files="GaRAGe_sample_raw.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

Generating train split: 1000 examples [00:00, 84894.63 examples/s]


In [5]:
from helper_functions_qa import (prepare_test_prompts, run_experiment, merge_df_into_dataset_by_order)

In [6]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [7]:
Qwen3_4B = "Qwen/Qwen3-4B"

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)

# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:12<00:00,  4.09s/it]


cuda


In [8]:
df = pd.read_json("GaRAGe_sample_raw.jsonl", lines=True)
test_prompts = prepare_test_prompts(df, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 1000
Generation complete: 1000 prompts
Average prompt length: 412 bytes (~103 tokens)

Analyze the following input user query:

{"query": "How does Amazon Cognito support passwordless authentication methods?"}

Please provide your analysis in the following JSON format:

{"query": "How does Amazon Cognito support passwordless authentication methods?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [9]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df)
test_df.to_csv('./intermediate/BASELINE_GaRAGe_classified.csv')
test_df

100%|██████████| 200/200 [1:45:14<00:00, 31.57s/it]


,id,request,answer,thinking,model_response,model_pred
0,76cc2729-2ec6-4100-b9c1-59c0a1b3d3e9,How does Amazon Cognito support passwordless a...,[Amazon Cognito supports passwordless authenti...,"<think>\nOkay, let's see. The user is asking h...","{\n ""query"": ""How does Amazon Cognito support...",fully specified
1,a881c11a-4ed7-4a9a-a4ee-87afcc427b65,What strategies can businesses employ to mitig...,"[To mitigate ASC 842 compliance challenges, bu...","<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What strategies can businesses ...",underspecified
2,38512574-e68d-40fd-ae97-2162db411df7,How does the Deep Retinal Convolution Neural N...,[The Deep Retinal Convolutional Neural Network...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""How does the Deep Retinal Con...",underspecified
3,ae94948d-eff9-4c5c-aa5e-b0f8131c7cc4,How does the integration of Radiata's technolo...,[The integration of Radiata's technology into ...,"<think>\nOkay, let me try to figure this out. ...","{\n ""query"": ""How does the integration of Rad...",fully specified
4,709d11b3-eb14-43be-9a17-d567eb7abd40,"How does Zoe Law's ""Legends"" exhibition reflec...","[The ""Legends"" exhibition by Zoë Law reflects ...","<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How does Zoe Law's \""Legends\"" ...",underspecified
...,...,...,...,...,...,...
995,154d2c6c-4fbb-49f1-8241-61ffc0b24d8b,What challenges does self-managed OpenSearch d...,[Self-managed OpenSearch deployments face seve...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What challenges does self-manag...",underspecified
996,e39aaa94-ce43-4b0a-8701-99d737a0d066,What are the implications of Cleveland-Cliffs ...,[Cleveland-Cliffs CEO's plan to make another o...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""What are the implications of Cl...",underspecified
997,1ca2aa68-7031-49b3-98d9-4cb52e06571c,How has the expansion of telehealth services u...,[The expansion of telehealth services under Me...,"<think>\nOkay, let's see. The user asked how t...","{\n ""query"": ""How has the expansion of telehe...",underspecified
998,d43f5fcc-797c-4af3-9779-995d505fd9bc,How did the US-brokered ceasefire agreement af...,[The US-brokered ceasefire agreement had a sig...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How did the US-brokered ceasefi...",fully specified


In [10]:
annotated_dataset = merge_df_into_dataset_by_order(
    GaRAGe_sample, 
    test_df, 
    columns=["thinking", "model_response", "model_pred"],
    prefix="qwen3_",
    inplace=False
)

In [11]:
# Create the UND subset
underspecified_set = annotated_dataset.filter(
    lambda x: x["qwen3_model_pred"].strip().lower() == "underspecified"
)

# Create the FS subset
fully_specified_set = annotated_dataset.filter(
    lambda x: x["qwen3_model_pred"].strip().lower() == "fully specified"
)

# The size of subsets
print(f"Underspecified samples: {len(underspecified_set)}")
print(f"Fully specified samples: {len(fully_specified_set)}")

Filter: 100%|██████████| 1000/1000 [00:00<00:00, 65677.62 examples/s]

Underspecified samples: 603
Fully specified samples: 397


In [12]:
annotated_dataset.to_json("./intermediate/BASELINE_classified_GaRAGe_sample_all.jsonl", orient="records", lines=True)
underspecified_set.to_json("./intermediate/BASELINE_classified_GaRAGe_sample_UND.jsonl", orient="records", lines=True)
fully_specified_set.to_json("./intermediate/BASELINE_classified_GaRAGe_sample_FS.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 71.35ba/s]


1701267

## QA implementations

### Setup

In [2]:
underspecified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_GaRAGe_sample_UND.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_GaRAGe_sample_FS.jsonl",
    split="all"
)

Generating train split: 603 examples [00:00, 56955.49 examples/s]
Generating train split: 397 examples [00:00, 69021.29 examples/s]


### Implementation

In [3]:
from openai import OpenAI
openai_api = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=openai_api)

In [4]:
from helper_functions_qa import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)

In [5]:
df_UND = pd.read_json('./intermediate/BASELINE_classified_GaRAGe_sample_UND.jsonl', lines=True)
df_FS = pd.read_json('./intermediate/BASELINE_classified_GaRAGe_sample_FS.jsonl', lines=True)

In [6]:
df_UND

,id,request,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred
0,a881c11a-4ed7-4a9a-a4ee-87afcc427b65,What strategies can businesses employ to mitig...,"[To mitigate ASC 842 compliance challenges, bu...","<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What strategies can businesses ...",underspecified
1,38512574-e68d-40fd-ae97-2162db411df7,How does the Deep Retinal Convolution Neural N...,[The Deep Retinal Convolutional Neural Network...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""How does the Deep Retinal Con...",underspecified
2,709d11b3-eb14-43be-9a17-d567eb7abd40,"How does Zoe Law's ""Legends"" exhibition reflec...","[The ""Legends"" exhibition by Zoë Law reflects ...","<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How does Zoe Law's \""Legends\"" ...",underspecified
3,19805270-2046-4431-9561-45d95e8b317b,How does the involvement of Tyco Ventures and ...,[The involvement of Tyco Ventures and Integral...,"<think>\nOkay, let's see. The user is asking h...","{\n ""query"": ""How does the involvement of Tyc...",underspecified
4,e59f3a34-a831-44a0-a379-dcb99fc4305c,How has the Drake-Kendrick Lamar feud influenc...,[The feud between Drake and Kendrick Lamar has...,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""How has the Drake-Kendrick La...",underspecified
...,...,...,...,...,...,...
598,a57d6b79-4f64-42ac-a304-1ad2244a194d,How did Elon Musk's opposition influence the g...,[Elon Musk's opposition led to the president-e...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How did Elon Musk's opposition ...",underspecified
599,154d2c6c-4fbb-49f1-8241-61ffc0b24d8b,What challenges does self-managed OpenSearch d...,[Self-managed OpenSearch deployments face seve...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What challenges does self-manag...",underspecified
600,e39aaa94-ce43-4b0a-8701-99d737a0d066,What are the implications of Cleveland-Cliffs ...,[Cleveland-Cliffs CEO's plan to make another o...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""What are the implications of Cl...",underspecified
601,1ca2aa68-7031-49b3-98d9-4cb52e06571c,How has the expansion of telehealth services u...,[The expansion of telehealth services under Me...,"<think>\nOkay, let's see. The user asked how t...","{\n ""query"": ""How has the expansion of telehe...",underspecified


In [7]:
df_FS

,id,request,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred
0,76cc2729-2ec6-4100-b9c1-59c0a1b3d3e9,How does Amazon Cognito support passwordless a...,[Amazon Cognito supports passwordless authenti...,"<think>\nOkay, let's see. The user is asking h...","{\n ""query"": ""How does Amazon Cognito support...",fully specified
1,ae94948d-eff9-4c5c-aa5e-b0f8131c7cc4,How does the integration of Radiata's technolo...,[The integration of Radiata's technology into ...,"<think>\nOkay, let me try to figure this out. ...","{\n ""query"": ""How does the integration of Rad...",fully specified
2,178c2659-4512-4fb7-882f-a47d7a756dcd,How do TMD factorization and collinear factori...,[TMD factorization and collinear factorization...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""How do TMD factorization and co...",fully specified
3,5551da4f-3468-48fe-895e-57213d573a2a,What advanced routing policies does Amazon Rou...,[Amazon Route 53 supports several advanced rou...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What advanced routing policies ...",fully specified
4,00140d51-5e44-417c-b14d-5102d1ce3e42,What are the key differences between Gemini 1....,[Gemini 1.5 Pro utilizes a transformer-based a...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What are the key differences be...",fully specified
...,...,...,...,...,...,...
392,7ad63dc9-235a-4fb4-9196-74552c0d6e3f,What implications do Meta's updated content po...,"[Meta's updated content policies, which replac...","<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""What implications do Meta's upd...",fully specified
393,5c372969-523f-42dd-893a-6fa33e1b1b0a,How did Donald Trump's campaign respond to the...,[Donald Trump's campaign responded to the assa...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How did Donald Trump's campaign...",fully specified
394,775d97fd-4033-4373-898e-9c222e7f180a,What are the top reader interests and policy p...,[There is not enough grounding for an answer.],"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What are the top reader interes...",fully specified
395,1d666b62-055e-46ea-abc7-9ea223e753f8,How does Abbott's Confirm Rx Insertable Cardia...,[Abbott's Confirm Rx Insertable Cardiac Monito...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How does Abbott's Confirm Rx In...",fully specified


In [8]:
df_UND = df_UND.rename(columns={"request": "question"})
df_FS = df_FS.rename(columns={"request": "question"})
df_UND.to_json('./intermediate/BASELINE_classified_GaRAGe_sample_UND.jsonl', orient="records", lines=True)
df_FS.to_json('./intermediate/BASELINE_classified_GaRAGe_sample_FS.jsonl', orient="records", lines=True)

underspecified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_GaRAGe_sample_UND.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_GaRAGe_sample_FS.jsonl",
    split="all"
)

Generating train split: 603 examples [00:00, 77160.45 examples/s]
Generating train split: 397 examples [00:00, 71751.57 examples/s]


In [9]:
short_results_UND = batch_QA_with_progress(
    underspecified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_short_answer",
    fill_value=["error"],
    client=client,
    model="gpt-4o-2024-11-20",
    temperature=0.0
)

Running model_short_answer: 100%|██████████| 61/61 [18:32<00:00, 18.23s/it]


In [10]:
# batch QA for FS
short_results_FS = batch_QA_with_progress(
    fully_specified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_short_answer",
    fill_value=["error"],
    client=client,
    model="gpt-4o-2024-11-20",
    temperature=0.0
)

Running model_short_answer: 100%|██████████| 40/40 [12:54<00:00, 19.35s/it]


In [11]:
qa_underspecified = deepcopy(underspecified_set)

for key in short_results_UND:
    qa_underspecified = qa_underspecified.add_column(key, short_results_UND[key])

qa_underspecified.to_json("./intermediate/BASELINE_GaRAGe_UND_qa_gpt.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 58.43ba/s]


2699815

In [12]:
qa_fully_specified = deepcopy(fully_specified_set)

for key in short_results_FS:
    qa_fully_specified = qa_fully_specified.add_column(key, short_results_FS[key])

qa_fully_specified.to_json("./intermediate/BASELINE_GaRAGe_FS_qa_gpt.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 100.13ba/s]


1814727

In [13]:
df = pd.read_json("./intermediate/BASELINE_GaRAGe_UND_qa_gpt.jsonl", lines=True)
df.to_csv('./output_csv/BASELINE_GaRAGe_UND_qa_gpt.csv')

df = pd.read_json("./intermediate/BASELINE_GaRAGe_FS_qa_gpt.jsonl", lines=True)
df.to_csv('./output_csv/BASELINE_GaRAGe_FS_qa_gpt.csv')

In [14]:
df

,id,question,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred,model_short_answer
0,76cc2729-2ec6-4100-b9c1-59c0a1b3d3e9,How does Amazon Cognito support passwordless a...,[Amazon Cognito supports passwordless authenti...,"<think>\nOkay, let's see. The user is asking h...","{\n ""query"": ""How does Amazon Cognito support...",fully specified,[Amazon Cognito supports passwordless authenti...
1,ae94948d-eff9-4c5c-aa5e-b0f8131c7cc4,How does the integration of Radiata's technolo...,[The integration of Radiata's technology into ...,"<think>\nOkay, let me try to figure this out. ...","{\n ""query"": ""How does the integration of Rad...",fully specified,[It enhances Cisco's wireless LAN product perf...
2,178c2659-4512-4fb7-882f-a47d7a756dcd,How do TMD factorization and collinear factori...,[TMD factorization and collinear factorization...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""How do TMD factorization and co...",fully specified,[- TMD factorization explains transverse singl...
3,5551da4f-3468-48fe-895e-57213d573a2a,What advanced routing policies does Amazon Rou...,[Amazon Route 53 supports several advanced rou...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What advanced routing policies ...",fully specified,[- Weighted routing \n- Latency-based routing...
4,00140d51-5e44-417c-b14d-5102d1ce3e42,What are the key differences between Gemini 1....,[Gemini 1.5 Pro utilizes a transformer-based a...,"<think>\nOkay, let's see. The user is asking a...","{\n ""query"": ""What are the key differences be...",fully specified,"[- ""Gemini 1.5 Flash"" focuses on faster proces..."
...,...,...,...,...,...,...,...
392,7ad63dc9-235a-4fb4-9196-74552c0d6e3f,What implications do Meta's updated content po...,"[Meta's updated content policies, which replac...","<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""What implications do Meta's upd...",fully specified,[Meta's updated content policies aim to foster...
393,5c372969-523f-42dd-893a-6fa33e1b1b0a,How did Donald Trump's campaign respond to the...,[Donald Trump's campaign responded to the assa...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How did Donald Trump's campaign...",fully specified,[- There is no verified record of an assassina...
394,775d97fd-4033-4373-898e-9c222e7f180a,What are the top reader interests and policy p...,[There is not enough grounding for an answer.],"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What are the top reader interes...",fully specified,"[Student mental health, Academic freedom, Dive..."
395,1d666b62-055e-46ea-abc7-9ea223e753f8,How does Abbott's Confirm Rx Insertable Cardia...,[Abbott's Confirm Rx Insertable Cardiac Monito...,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""How does Abbott's Confirm Rx In...",fully specified,[- It continuously monitors and records heart ...


## Evaluations

In [15]:
underspecified_set_qa = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_GaRAGe_UND_qa_gpt.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set_qa = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_GaRAGe_FS_qa_gpt.jsonl",
    split="all"
)

Generating train split: 603 examples [00:00, 53588.55 examples/s]
Generating train split: 397 examples [00:00, 62187.73 examples/s]


### Squad EM + F1

In [16]:
from helper_functions_qa import evaluate_squad_per_sample_multi_ref_pred

In [18]:
# Official squad script for avg EM and F1, not possible for t-test


# Evaluate fully specified subset
dataset = load_dataset("json", data_files="./intermediate/BASELINE_GaRAGe_FS_qa_gpt.jsonl", split="all")

# 加载 HuggingFace 的 squad 评估器
squad_metric = evaluate.load("squad")

# 构造 predictions 和 references（标准格式）
predictions = [
    {
        "id": str(i),
        "prediction_text": pred[0] if isinstance(pred, list) and pred else ""
    }
    for i, pred in enumerate(dataset["model_short_answer"])
]

references = [
    {
        "id": str(i),
        "answers": {
            "text": ref if isinstance(ref, list) else [ref],
            "answer_start": [0] * len(ref if isinstance(ref, list) else [ref])
        }
    }
    for i, ref in enumerate(dataset["answer"])
]

# 计算 SQuAD-style EM 和 F1
results = squad_metric.compute(predictions=predictions, references=references)

# 打印平均指标
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")

Exact Match: 0.00
F1 Score: 14.55


In [19]:
# Official squad script for avg EM and F1, not possible for t-test


# Evaluate fully specified subset
dataset = load_dataset("json", data_files="./intermediate/BASELINE_GaRAGe_UND_qa_gpt.jsonl", split="all")

# 加载 HuggingFace 的 squad 评估器
squad_metric = evaluate.load("squad")

# 构造 predictions 和 references（标准格式）
predictions = [
    {
        "id": str(i),
        "prediction_text": pred[0] if isinstance(pred, list) and pred else ""
    }
    for i, pred in enumerate(dataset["model_short_answer"])
]

references = [
    {
        "id": str(i),
        "answers": {
            "text": ref if isinstance(ref, list) else [ref],
            "answer_start": [0] * len(ref if isinstance(ref, list) else [ref])
        }
    }
    for i, ref in enumerate(dataset["answer"])
]

# 计算 SQuAD-style EM 和 F1
results = squad_metric.compute(predictions=predictions, references=references)

# 打印平均指标
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")

Exact Match: 0.00
F1 Score: 12.36


In [20]:
squad_scored_UND, UND_f1_list, UND_em_list = evaluate_squad_per_sample_multi_ref_pred(underspecified_set_qa, ref_col="answer")
squad_scored_UND.to_json("./intermediate/BASELINE_GaRAGe_UND_qa_gpt_with_squad_scores.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 67.84ba/s]


2713920

In [21]:
squad_scored_FS, FS_f1_list, FS_em_list = evaluate_squad_per_sample_multi_ref_pred(fully_specified_set_qa, ref_col="answer")
squad_scored_FS.to_json("./intermediate/BASELINE_GaRAGe_FS_qa_gpt_with_squad_scores.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 109.25ba/s]


1824204

In [22]:
df = pd.read_json("./intermediate/BASELINE_GaRAGe_UND_qa_gpt_with_squad_scores.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_GaRAGe_UND_qa_gpt_with_squad_scores.csv')

df = pd.read_json("./intermediate/BASELINE_GaRAGe_FS_qa_gpt_with_squad_scores.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_GaRAGe_FS_qa_gpt_with_squad_scores.csv')

In [23]:
UND_mean_em = np.mean(UND_em_list)  # em_scores: EM list per sample
UND_mean_f1 = np.mean(UND_f1_list)  # f1_scores F1 list per sample
print(f"UND Exact Match (avg): {UND_mean_em * 100:.2f}")
print(f"UND F1 Score (avg): {UND_mean_f1 * 100:.2f}")

FS_mean_em = np.mean(FS_em_list)  # em_scores: EM list per sample
FS_mean_f1 = np.mean(FS_f1_list)  # f1_scores F1 list per sample
print(f"FS Exact Match (avg): {FS_mean_em * 100:.2f}")
print(f"FS F1 Score (avg): {FS_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(FS_f1_list, UND_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(FS_em_list, UND_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

UND Exact Match (avg): 0.00
UND F1 Score (avg): 12.77
FS Exact Match (avg): 0.00
FS F1 Score (avg): 14.77
F1: t=3.430, p=0.0006
EM: t=nan, p=nan


## Ragas

In [24]:
from helper_functions_qa import answer_accuracy

In [25]:
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

UND_full = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_GaRAGe_UND_qa_gpt_with_squad_scores.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

FS_full = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_GaRAGe_FS_qa_gpt_with_squad_scores.jsonl",
    split="all"
)

Generating train split: 603 examples [00:00, 82947.93 examples/s]
Generating train split: 397 examples [00:00, 64915.16 examples/s]


In [26]:
UND_ragas = await answer_accuracy(UND_full, evaluator_llm, ref_col = "answer")
UND_ragas.to_csv("./output_csv/GaRAGe_UND_gpt4o_Ragas.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 16.94ba/s]


2631186

In [27]:
FS_ragas = await answer_accuracy(FS_full, evaluator_llm, ref_col = "answer")
FS_ragas.to_csv("./output_csv/GaRAGe_FS_gpt4o_Ragas.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 26.51ba/s]


1769797

In [28]:
UND_ragas_AA = list(UND_ragas["ragas_AA_short"])
FS_ragas_AA = list(FS_ragas["ragas_AA_short"])

UND_mean_AA = np.mean(UND_ragas_AA)
print(f"UND AA (avg): {UND_mean_AA * 100:.2f}")


FS_mean_AA = np.mean(FS_ragas_AA)
print(f"FS AA (avg): {FS_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(FS_ragas_AA, UND_ragas_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

UND AA (avg): 49.50
FS AA (avg): 53.72
AA: t=1.611, p=0.1075
